# Laboratorio 10 - Visión Por Computadora

Autores:

- Nelson García
- Joaquín Puente
- Diego Linares

Link al repositorio: https://github.com/Its-Japo/VisionXComputadora/tree/main/Lab_10

# Task 1

En clase explicamos que la Difusión "esculpe" la imagen restando ruido de un tensor dentro del Espacio
Latente, paso a paso, antes de que el Decoder la convierta en píxeles. Usted deberá demostrar este
comportamiento interrumpiendo el proceso para observar la evolución temporal.

Instalar librerías necesarias:

In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors huggingface_hub matplotlib pillow

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Activa GPU en Colab: Entorno de ejecución > Cambiar tipo de entorno > GPU")

In [ ]:
#   Cargar Stable Diffusion 1.5 en GPU

import torch
from diffusers import StableDiffusionPipeline
from pathlib import Path
import matplotlib.pyplot as plt

device = "cuda"
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True
)

pipe = pipe.to(device)

# Ayuda a reducir memoria en Colab.
pipe.enable_attention_slicing()

print("Modelo cargado en:", pipe.device)
print("VAE scaling factor:", pipe.vae.config.scaling_factor)

In [ ]:
#Interceptar los latentes en los pasos 4, 10, 16 y 20

prompt = "A highly detailed cinematic and futuristic fruit glowing in a cyberpunk laboratory, neon lights, 4k resolution"

num_inference_steps = 20
seed = 2026

# Se fija la semilla global y también el generador CUDA.
torch.manual_seed(seed)
generator = torch.Generator(device=device).manual_seed(seed)

capture_steps = {4, 10, 16, 20}
latents_by_step = {}

def capture_latents_callback(pipe, step_index, timestep, callback_kwargs):
    """
    step_index viene indexado desde 0.
    Por eso step_number = step_index + 1.
    """
    step_number = step_index + 1

    if step_number in capture_steps:
        latents = callback_kwargs["latents"]

        # Guardamos copia exacta en CPU para no modificar el proceso.
        latents_by_step[step_number] = latents.detach().clone().cpu()

        print(
            f"Latente guardado en paso {step_number:02d} | "
            f"shape={tuple(latents.shape)} | dtype={latents.dtype}"
        )

    return callback_kwargs

In [ ]:
#Ejecutar el pipeline sin decodificar automáticamente la imagen final

with torch.inference_mode():
    _ = pipe(
        prompt=prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=7.5,
        generator=generator,
        callback_on_step_end=capture_latents_callback,
        callback_on_step_end_tensor_inputs=["latents"],
        output_type="latent"
    )

print("Pasos capturados:", sorted(latents_by_step.keys()))

In [ ]:
#Decodificar manualmente cada latente

def decode_latent_to_pil(latent_cpu, pipe):
    """
    Recibe un tensor latente guardado en CPU.
    Lo mueve a GPU, aplica el scaling_factor correcto,
    lo decodifica con el VAE y lo convierte a PIL.
    """
    latent = latent_cpu.to(device=pipe.device, dtype=pipe.vae.dtype)

    # Desescalado requerido antes del VAE.
    latent = latent / pipe.vae.config.scaling_factor

    with torch.inference_mode():
        image_tensor = pipe.vae.decode(latent, return_dict=False)[0]

    # Convierte tensor [-1, 1] a imagen PIL.
    image_pil = pipe.image_processor.postprocess(
        image_tensor,
        output_type="pil"
    )[0]

    return image_pil

In [ ]:
out_dir = Path("resultados_task_1")
out_dir.mkdir(exist_ok=True)

decoded_images = {}

for step in [4, 10, 16, 20]:
    img = decode_latent_to_pil(latents_by_step[step], pipe)
    decoded_images[step] = img

    save_path = out_dir / f"latente_decodificado_paso_{step:02d}.png"
    img.save(save_path)

    print("Guardada:", save_path)

Mostrar resultados:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, step in zip(axes, [4, 10, 16, 20]):
    ax.imshow(decoded_images[step])
    ax.set_title(f"Paso {step}/20")
    ax.axis("off")

plt.suptitle("Evolución temporal del denoising en el espacio latente", fontsize=16)
plt.tight_layout()

grid_path = out_dir / "grid_evolucion_latentes.png"
plt.savefig(grid_path, dpi=150, bbox_inches="tight")
plt.show()

print("Cuadrícula guardada en:", grid_path)

# Task 2
Como vimos al final de la sesión, el modelo Nano Banana de Google representa la vanguardia en
eficiencia: usar técnicas de destilación para saltarse pasos matemáticos de la Cadena de Markov y correr
modelos generativos en milisegundos. Usted simulará este escenario midiendo empíricamente el trade-off
(costo-beneficio) en producción.